In [15]:
import polars as pl

In [16]:
books_df = pl.scan_ndjson("../processed-data/cleaned_books_fantasy_paranormal.json")
interactions_df = pl.scan_ndjson("../processed-data/cleaned_interactions_fantasy_paranormal.json")
authors_df = pl.scan_ndjson("../raw-data/goodreads_book_authors.json")
series_df = pl.scan_ndjson("../raw-data/goodreads_book_series.json")

In [17]:
# filter out descriptions with fewer than 50 words
books_df = books_df.filter(pl.col('description').str.count_matches(r"\w+") >= 50)

In [18]:
books_df.select('language_code').unique().show()

language_code
str
"""ita"""
"""swe"""
"""nob"""
"""frs"""
"""ast"""


In [19]:
# Keep only english variations
# """en"""
# """en-CA"""
# """en-GB"""
# """en-US"""
# """eng"""

books_df = books_df.filter(
    pl.col('language_code').is_in(['en', 'en-CA', 'en-GB', 'en-US', 'eng'])
)


In [20]:
books_df = books_df.explode("authors").with_columns(
    pl.col("authors").struct.field("author_id").alias("author_id")
)
books_df = books_df.join(
    authors_df.select(["author_id", "name"]),
    on="author_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["authors", "author_id", "name"]).first(),
    pl.col("name").drop_nulls().alias("author_names")
)


In [21]:
books_df = books_df.explode("series").with_columns(
    pl.col("series").alias("series_id")
)
books_df = books_df.join(
    series_df.select(["series_id", pl.col("title").alias("series_title")]),
    on="series_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["series", "series_id", "series_title"]).first(),
    pl.col("series_title").drop_nulls().alias("series")
)


In [22]:
books_df = books_df.with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().struct.field("name")
    ).alias("popular_shelves")
).with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().filter(
            ~pl.element().str.contains(r"(?i)read|own|buy|fav|library|audio|kindle|ebook")
        )
    )
)


In [23]:
books_df.select('language_code').unique().show()

language_code
str
"""en-GB"""
"""eng"""
"""en-CA"""
"""en"""
"""en-US"""


In [10]:
books_df.head().show()

book_id,isbn,text_reviews_count,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title,title_without_series,author_names,series
i64,str,i64,str,str,list[str],str,str,f64,str,list[str],str,str,str,str,i64,i64,str,i64,str,i64,str,str,i64,i64,str,str,list[str],list[str]
18734992,"""1937053911""",8,"""US""","""eng""","[""fantasy"", ""giveaways"", … ""potentially-interesting""]","""""","""false""",4.29,"""B00JPFMTZK""","[""10783236"", ""17572336"", … ""13268319""]","""As the littlest Hapenny, a rac…","""Paperback""","""https://www.goodreads.com/book…","""Spencer Hill Press""",160,15,"""9781937053918""",4,"""""",2014,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",19,17987646,"""Hapenny Magick""","""Hapenny Magick""","[""Jennifer Carson""]","[""Hapenny Magick""]"
15729469,"""""",2,"""US""","""eng""","[""romance"", ""paranormal"", … ""ward""]","""B002VA9GXE""","""false""",4.21,"""""","[""715791"", ""4497978"", … ""1599297""]","""(LENGTH 13 hrs and 35 mins) Th…","""Audiobook""","""https://www.goodreads.com/book…","""Recorded Books""",null,null,"""""",1,"""Unabridged - Audible Edition""",2009,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",12,2158128,"""Dark Lover""","""Dark Lover""","[""J.R. Ward"", ""Jim Frangione""]","[""Black Dagger Brotherhood""]"
6571580,"""""",7,"""US""","""eng""","[""young-adult"", ""paranormal"", … ""ghost-stories""]","""B000FC2OIE""","""true""",4.16,"""B000FC2OIE""","[""12871657"", ""2298960"", … ""127439""]","""What - or who - is buried in S…","""""","""https://www.goodreads.com/book…","""HarperTeen""",336,6,"""""",10,"""""",2009,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",229,2013708,"""Darkest Hour (The Mediator, #4…","""Darkest Hour (The Mediator, #4…","[""Meg Cabot""]","[""The Mediator""]"
26568733,"""""",2,"""US""","""eng""","[""paranormal"", ""paranormal-romance"", … ""to-search-for""]","""B0150BUW2E""","""true""",4.31,"""B0150BUW2E""","[""28454804"", ""17929641"", … ""13793030""]","""Recommended for 18+ Erotic Sha…","""""","""https://www.goodreads.com/book…","""""",null,null,"""""",null,"""""",null,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",30,41118574,"""Hawk (Samuel's Pride, #6)""","""Hawk (Samuel's Pride, #6)""","[""Kathi S. Barton""]","[""Samuel's Pride""]"
18523167,"""1930076215""",10,"""US""","""eng""","[""need"", ""books-to-review"", … ""arc""]","""""","""false""",3.92,"""""",[],"""The town of Wickerwire, Iowa h…","""Paperback""","""https://www.goodreads.com/book…","""Wigwam Publishing""",278,1,"""9781930076211""",10,"""""",2013,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",13,26225911,"""All That Glitters""","""All That Glitters""","[""Jackie Sonnenberg""]",[]


In [11]:
books_df = books_df.drop([
    'isbn', 'text_reviews_count', 'country_code', 'language_code',
    'is_ebook', 'kindle_asin', 'format', 'num_pages',
    'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'image_url',
    'title_without_series', 'ratings_count', 'publisher', 'similar_books', 'book_id', 'asin'
])
books_df.head().show()


popular_shelves,average_rating,description,link,url,work_id,title,author_names,series
list[str],f64,str,str,str,i64,str,list[str],list[str]
"[""rpg"", ""gaming"", … ""dungeons-and-dragons-4th-ed""]",3.81,"""The best way to start playing …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",18968641,"""Dungeons & Dragons Fantasy Rol…","[""Bill Slavicsek"", ""Rodney Thompson"", … ""Ralph Horsley""]",[]
"[""urban-fantasy"", ""science-fiction"", … ""dystopia""]",3.63,"""An alternate cover for this IS…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",43082719,"""The Drafter (The Peri Reed Chr…","[""Kim Harrison""]","[""The Peri Reed Chronicles""]"
"[""urban-fantasy"", ""fantasy"", … ""war""]",3.97,"""Kitty has been tapped as the k…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",15498308,"""Kitty Steals the Show (Kitty N…","[""Carrie Vaughn""]","[""Kitty Norville""]"
"[""horror"", ""fantasy"", … ""traditionally-published""]",3.73,"""Strange things exist on the pe…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",18483999,"""The Croning""","[""Laird Barron""]",[]
"[""guardians-of-the-word"", ""series"", … ""fantasy-romance""]",4.5,"""There are no holes in space. N…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",21467001,"""Union (Guardians of the Word, …","[""Jolea M. Harrison""]","[""Guardians of the Word""]"


In [12]:
books_df = books_df.with_columns(
    pl.concat_str(
        [
            pl.col("title").fill_null(""),
            pl.col("author_names").list.join(", ").fill_null(""),
            pl.col("series").list.join(", ").fill_null(""),
            pl.col("popular_shelves").list.join(", ").fill_null(""),
            pl.col("description").fill_null("")
        ],
        separator=" "
    ).alias("combined_text")
)


In [13]:
# Basic text preprocessing on combined_text
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))
stopwords_regex = r"(?i)\b(" + "|".join(STOPWORDS) + r")\b"

books_df = books_df.with_columns(
    pl.col("combined_text")
    .str.to_lowercase()
    .str.replace_all(r"[^\w\s]", " ") # remove punctuation
    .str.replace_all(stopwords_regex, "") # remove stopwords
    .str.replace_all(r"\s+", " ") # remove multiple spaces
    .str.strip_chars() # trim leading/trailing spaces
)

books_df.head().show()


popular_shelves,average_rating,description,link,url,work_id,title,author_names,series,combined_text
list[str],f64,str,str,str,i64,str,list[str],list[str],str
"[""science-fiction"", ""sci-fi"", … ""skimmed-it""]",3.61,"""For one researcher, Triplet is…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",209541,"""Triplet""","[""Timothy Zahn""]",[],"""triplet timothy zahn science f…"
"[""horror"", ""fiction"", … ""demons""]",3.82,"""Experience the unspeakably evi…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",3059336,"""The Damnation Game""","[""Clive Barker""]",[],"""damnation game clive barker ho…"
"[""m-m"", ""paranormal"", … ""just-not-convinced-yet""]",3.64,"""Baking the best cupcakes in Sa…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",14685493,"""The Cake (Elemental Superpower…","[""Serena Yates"", ""A.J. Llewellyn""]","[""Elemental Superpowers""]","""cake elemental superpowers 1 s…"
"[""vampires"", ""paranormal-romance"", … ""my-books""]",4.15,"""The night is not safe for mort…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",6579128,"""First Drop of Crimson (Night H…","[""Jeaniene Frost""]","[""Night Huntress Universe"", ""Night Huntress World""]","""first drop crimson night huntr…"
"[""fantasy"", ""middle-grade"", … ""kid-books""]",3.8,"""Rosemary Bliss's family has a …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",15554388,"""Bliss (The Bliss Bakery, #1)""","[""Kathryn Littlewood""]","[""The Bliss Bakery""]","""bliss bliss bakery 1 kathryn l…"


In [14]:
import os
os.makedirs("../processed-data", exist_ok=True)
books_df.collect().write_ndjson("../processed-data/processed_books_texts.json")
print("Dataset successfully saved!")

Dataset successfully saved!
